### Imports

In [16]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

### Read files

In [3]:
X_train = pd.read_csv('X_train.csv',index_col='ROW_ID')
y_train = pd.read_csv('y_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('X_test.csv',index_col='ROW_ID')

### Initial exploration

In [ ]:
print(X_train.head(3)) # print the data structure to double check, some NaN are present

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
0       DATE_0001  ALLOCATION_01 -0.018192 -0.000306 -0.006881 -0.002393   
1       DATE_0001  ALLOCATION_02 -0.006394 -0.001059  0.001565  0.000033   
2       DATE_0001  ALLOCATION_03 -0.016587 -0.004517 -0.005306  0.004314   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
0       0.000507 -0.001270 -0.002539  0.002830  ...         0.818730   
1       0.002829  0.001725  0.000875 -0.002160  ...        -1.390336   
2       0.006471 -0.005868 -0.005030 -0.001488  ...         0.961318   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
0              0.941014         0.714129        -0.323847         0.525097   
1             -0.651784 

(None, None)

In [ ]:
print(X_test.head(3)) # same situations

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
527073  DATE_2523  ALLOCATION_01 -0.000523  0.000066 -0.001601  0.002030   
527074  DATE_2523  ALLOCATION_02  0.002868 -0.000870  0.000609  0.003156   
527075  DATE_2523  ALLOCATION_03 -0.000435  0.003141  0.001410 -0.011531   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
527073 -0.000137 -0.001506  0.000479 -0.001006  ...        -1.124653   
527074  0.001140  0.000728  0.002079  0.000126  ...        -1.310546   
527075 -0.000685  0.002088 -0.003898  0.008467  ...         1.402281   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
527073        -1.263611        -1.177919        -1.251840        -1.242161   
527074        -0.925154 

In [ ]:
print(y_train.head(3)) # fast print check on target data structure

          target
ROW_ID          
0       0.009210
1       0.000405
2       0.005967


In [ ]:
print(X_train.shape, X_test.shape) # look at the shape of the data
print("portfolios:", X_train["ALLOCATION"].nunique(), X_test["ALLOCATION"].nunique()) # 
# we have the same number of portfolio 278 and same number of columns in each database, good.

(527073, 44) (31870, 44)
portfolios: 278 278


In [14]:
X_train["target"] = y_train["target"]      # we attach the next-day return to each row
print("share of up days:", (X_train["target"] > 0).mean().round(4)) # we see how often we have up and down days

share of up days: 0.5072


In [ ]:
RET = [f"RET_{i}" for i in range(1, 21)]   # RET_1 = most recent day
# look at the number of Nan, i.e., missing days. We will have to deal with them
print("missing RET values: train", X_train[RET].isna().sum().sum(), " test", X_test[RET].isna().sum().sum()) 

missing RET values: train 262  test 294
